# Autoencoders & Variational Autoencoders (VAE)

A plain autoencoder compresses then reconstruct. It memorizes. It does not generate. Add one trick -- force the code to look Gaussian -- and you get sampler. That single trick, the reparameterization of `z = miu + sigmal * epsilon` is why every latent-diffusion and flow-matching image model you use in 2026 has a VAE at the input.

## Problem Definition

Compress a 784-pixel MINIST digit to a 16-number code. Then reconstruct. A plain autoencoder will ace reconstruction MSE but the code space is a lumpy mess. Pick a random point in the code space, decode it, and you get noise. It has no sampler, it is a compression model dressed up.

What you actually want is: 
1. The code space is a clean, smooth distribution you can sample from -- say an isotropic Gaussian `N(0, I)`.
2. Decoding any sample produces a plausible digit
3. The encoder and decoder still compress well. Three goals, one architecture, one loss.

VAE solve this by training the coder to output a distribution `q(z|x) = N(miu(x), sigma(x)^2)`, pulling that distribution toward the prior `N(0, I)` via a KL penalty, and then sampling `z` from `q(z|x)` before encoding. At inference time, drop the encoder, sample `z ~ N(0, I)`, decode, The KL penalty is what forces the code space to be structured.

In 2026 VAEs rarely ship standalone -- they have been outclassed by diffusion for raw image quality -- but they are the encoder of choic for every latent-diffusion model. Learn the VAE and you learn the invisible first layer of every imange pipeline you use.

## Basic Concept

Autoencoder vs VAE: the reparametrizaion trick.

### Autoencoder

`z = encoder(x), x_hat = decoder(z)`, loss = `||x - x_hat||^2`. Code space unstructured.

### VAE encoder

Outputs two vectors: `miu(x)` and `log(sigma(x)^2)`. These define `q(z|x) = N(miu, diag(sigma^2))`

### Reparameterizatoin trick

Sampling from `q(z|x)` is not differentialable. Rewrite the sample as `z = miu + sigma * epsilon` where `epsilon ~ N(0, I)`. Now z is a deterministic function of `(miu, sigma)` plus a non-parameter noise -- gradients flow through `miu` and `sigma`.

### Loss

Evidence Lower BOund (ELBO)
```
loss = reconstruction + beta * KL[q(z|x) || N(0, I)]
     = ||x - x_hat||^2 + beta * sum_i(sigma_i^2 + miu_i^2 - log(simga_i^2) - 1) / 2
```

Reconstruction pushes `x_hat` toward `x`. KL pushes `q(z|x)` toward the prior. They trade off. Small `beta <1` means sharper smaples, code space less Guassion. Large `beta > 1` means cleaner code space, blurrier samples. 

### Samping

At inference, draw `z ~ N(0, I)`, forward throgh decoder. On forward pass, no iterative sampling like diffusion.

# Build your Own

In [ ]:
import math
import random

def matmul(W, x):
    return [
        sum(W[i][j] * x[j] for j in range(len(x)))
        for i in range(len(W))
    ]

def add(a, b):
    return [x + y for x, y in zip(a, b)]

def tanh(v):
    return [math.tanh(x) for x in v]

def tanh_grad(v):
    return [1 - x**2 for x in v]

def randn_matrix(rows, cols, rng, scale=0.2):
    return [
        [rng.gauss(0, scale) for _ in range(cols)]
        for _ in rows
    ]

def init_vae(in_dim, hidden, z_dim, rng):
    return {
        "enc": {
            "W1": randn_matrix(hidden, in_dim, rng),
            "b1": [0.0] * hidden,
            "W_mu": randn_matrix(z_dim, hidden, rng),
            "b_mu": [0.0] * z_dim,
            "W_sig": randn_matrix(z_dim, hidden, rng),
            "b_sig": [0.0] * z_dim,
        },
        "dec": {
            "W1": randn_matrix(hidden, z_dim, rng),
            "b1": [0.0] * hidden,
            "W_out": randn_matrix(in_dim, hidden, rng),
            "b_out": [0.0] * in_dim
        }
    }

def clamp(v, lo, hi):
    return [min(hi, max(lo, x)) for x in v]

def forward(x, params, eps):
    enc, dec = params["enc"], params["dec"]
    h_enc = tanh(add(matmul(enc["W1"], x), enc["b1"]))
    mu = add(matmul(enc["W_mu"], h_enc), enc["b_mu"])
    log_sigma_sq = clamp(add(matmul(enc["W_sig"], h_enc), enc["b_sig"]), -6, 6)

    sigma = [math.exp(math.exp(0.5 * lv) ) for lv in log_sigma_sq]

    z = [m + s * e for m, s, e in zip(mu, sigma, eps)]

    h_dec = tanh(add(matmul(dec["W1"], z), dec["b1"]))
    x_hat = add(matmul(dec["W_out"], h_dec), dec["b_out"])

    return {
        "h_enc": h_enc,
        "mu": mu,
        "log_sigma_sq": log_sigma_sq,
        "sigma": sigma,
        "z": z,
        "h_dec": h_dec,
        "x_hat": x_hat,
    }

def loss_value(x, fwd, beta):
    recon = sum((a - b)** 2 for a, b in zip(x, fwd["x_hat"]))
    kl = 0.5 * sum(math.exp(lv) + m * m - lv - 1 for m, lv in zip(fwd["mu"], fwd["log_sigma_sq"]))
    return recon + beta * kl, recon, kl